# Deep-Dive Video Note Taker — Experimentation Notebook

This notebook is used to experiment with individual pipeline stages:
- Audio extraction
- Whisper transcription
- Text chunking
- LLM summarization
- RAG pipeline
- Action item extraction

In [ ]:
import sys
sys.path.insert(0, '..')

# Load environment
from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded ✅')

## 1. Audio Extraction

In [ ]:
from backend.services.audio_extractor import AudioExtractor

extractor = AudioExtractor()

# Replace with your video path
VIDEO_PATH = '../data/videos/sample.mp4'
JOB_ID     = 'experiment_001'

audio_path = extractor.extract(VIDEO_PATH, JOB_ID)
duration   = extractor.get_video_duration(VIDEO_PATH)

print(f'Audio extracted: {audio_path}')
print(f'Duration: {duration:.1f}s')

## 2. Whisper Transcription

In [ ]:
from backend.services.whisper_transcriber import WhisperTranscriber

transcriber = WhisperTranscriber()
transcript  = transcriber.transcribe(audio_path, JOB_ID)

print(f"Language: {transcript['language']}")
print(f"Segments: {len(transcript['segments'])}")
print(f"Preview: {transcript['text'][:300]}")

## 3. Text Chunking

In [ ]:
from backend.services.text_chunker import TextChunker

chunker = TextChunker(max_chunk_size=800, overlap=100)
chunks  = chunker.chunk_transcript(transcript)

print(f'Total chunks: {len(chunks)}')
for c in chunks[:3]:
    print(f"  Chunk {c['chunk_id']}: [{c['start_ts']} → {c['end_ts']}] — {len(c['text'].split())} words")

## 4. LLM Summarization

In [ ]:
from backend.services.summarizer import Summarizer

summarizer = Summarizer()

# Summarize just the first 3 chunks for experiment
test_chunks = chunks[:3]
summarized  = summarizer.summarize_chunks(test_chunks)

for s in summarized:
    print(f"\n=== Chunk {s['chunk_id']} [{s['start_ts']}] ===")
    print(s['summary'])

## 5. RAG Pipeline

In [ ]:
from backend.services.rag_pipeline import RAGPipeline

rag = RAGPipeline()
rag.index_chunks(chunks)

# Test a query
query   = 'What are the main topics discussed?'
results = rag.query(query, top_k=3)

print(f'Query: {query}')
for r in results:
    print(f"  Score: {r['score']:.3f} | [{r['start_ts']}] {r['text'][:100]}...")

## 6. Action Item Extraction

In [ ]:
from backend.services.action_item_extractor import ActionItemExtractor

extractor2   = ActionItemExtractor()
action_items = extractor2.extract(chunks[:5])

print(f'Found {len(action_items)} action items:')
for item in action_items:
    print(f"  [{item['type'].upper()}] [{item['priority']}] {item['description']} (@ {item['timestamp']})")

## 7. Full Pipeline Run

In [ ]:
from backend.services.timestamp_mapper import TimestampMapper
from backend.services.note_generator   import NoteGenerator

# Summarize all chunks
all_summarized = summarizer.summarize_chunks(chunks)
final_notes    = summarizer.generate_final_notes(all_summarized)

# Map timestamps
mapper     = TimestampMapper()
highlights = mapper.map_timestamps(all_summarized)
chapters   = mapper.generate_chapter_markers(highlights)

# Extract all action items
all_actions = extractor2.extract(chunks)

# Generate final notes
generator = NoteGenerator()
result    = generator.generate(
    job_id=JOB_ID,
    filename='sample.mp4',
    transcript=transcript,
    summarized_chunks=all_summarized,
    final_notes=final_notes,
    highlights=highlights,
    action_items=all_actions,
    chapters=chapters,
    duration=duration,
)

print('Notes saved to:', result['markdown_path'])
print('\n--- Preview (first 500 chars) ---')
print(result['markdown'][:500])